# Seasonal observing plan

This is the main planning notebook. Change the dates, providers, observatory, and magnitude limit in the first code cell, then run the cells in order. It produces target tables, a sky map, nightly visibility plots, visibility and priority tables as CSV+PNG, light-curve PNGs, and one PDF.

In [1]:
from pathlib import Path
import pandas as pd
from target_selection.planning import (load_candidates, collect_photometry, apply_magnitude_limit, save_visibility_products, save_priority_products, save_target_sky_map, save_planning_pdf)

START_DATE = '2026-10-07'
END_DATE = '2026-10-15'
OBSERVATORY = 'El Leoncito'
PROVIDERS = ('mop', 'opm')
MAGNITUDE_MAX = 17.0
MIN_ALTITUDE_DEG = 40.0
MIN_VISIBLE_MINUTES = 90.0
LOAD_PHOTOMETRY = True
MAKE_LIGHTCURVES = True
MAKE_LSST_CUTOUTS = False
OUTPUT = Path('outputs/seasonal_plan')
CACHE = OUTPUT / 'cache'
OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Load candidates from the selected TOMs

In [2]:
targets = load_candidates(PROVIDERS, start_date=START_DATE, end_date=END_DATE, include_photometry=LOAD_PHOTOMETRY, cache_dir=CACHE)
targets.to_csv(OUTPUT / 'targets_loaded.csv', index=False)
display(targets.head())

[targets] Querying MOP visibility for 2026-10-07 to 2026-10-15.
[targets] MOP targets with event data: 39
[planning] mop: 39 targets
[planning] opm: 185 targets


,target_id,RA,Dec,Site,Min airmass,mag_now,tap_priority,tap_priority_longte,mop_link,observation_date,...,ANTARES microlensing filter,Peak Planet Probability from fit ψ Rescaled,Bogus ALeRCE stamp,opm_updated,opm_target_links,opm_target_page_id,opm_target_url,magnitude_last,band_last,category
0,OGLE-2024-BLG-0808,18h24m19.2s,-17d04m27.5s,(El Leoncito) El Leoncito,1.2,16.582524,0.67726,54.49391,https://mop.lco.global/targets/45864/,2026-10-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OGLE-2025-BLG-0271,17h54m29.54s,-39d44m25.3s,(El Leoncito) El Leoncito,1.2,17.005745,1.08405,73.50817,https://mop.lco.global/targets/53412/,2026-10-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OGLE-2026-GD-0001,16h28m35.32s,-45d16m14.5s,(El Leoncito) El Leoncito,1.5,17.080133,1.31713,23.55821,https://mop.lco.global/targets/56074/,2026-10-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ZTF26aazuuhy,16h54m07.14187368s,-25d25m41.02903545s,(El Leoncito) El Leoncito,1.5,17.115612,0.51916,5.18471,https://mop.lco.global/targets/56889/,2026-10-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,OGLE-2025-BLG-0242,17h13m08.02s,-32d15m43.3s,(El Leoncito) El Leoncito,1.3,17.133847,1.83281,89.43,https://mop.lco.global/targets/53379/,2026-10-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Collect TOM and complementary-survey photometry

In [ ]:
observations = collect_photometry(targets, providers=PROVIDERS, include_associated_surveys=True, cache_dir=CACHE / 'photometry') if LOAD_PHOTOMETRY else pd.DataFrame()
observations.to_csv(OUTPUT / 'observations.csv', index=False)
print(f'Photometry rows: {len(observations):,}')

[planning] Photometry unavailable from mop: Provider 'mop' does not implement photometry.


## 3. Apply the magnitude filter before the visibility calculation

In [ ]:
candidates = apply_magnitude_limit(targets, MAGNITUDE_MAX)
candidates.to_csv(OUTPUT / 'targets_after_magnitude.csv', index=False)
print(f'Candidates below magnitude limit: {len(candidates):,}/{len(targets):,}')

## 4. Evaluate visibility and save nightly/summary products

In [ ]:
visibility_by_night, visibility_summary, visibility_plots, visibility_png = save_visibility_products(candidates, OUTPUT / 'visibility', start_date=START_DATE, end_date=END_DATE, observatory=OBSERVATORY, minimum_altitude_deg=MIN_ALTITUDE_DEG, minimum_observable_minutes=MIN_VISIBLE_MINUTES)
selected_ids = set(visibility_summary.loc[visibility_summary['n_visible_nights'].gt(0), 'target_id'])
selected = candidates.loc[candidates['target_id'].isin(selected_ids)].copy()
selected.to_csv(OUTPUT / 'selected_targets.csv', index=False)
display(visibility_summary.head())

## 5. Priority table and sky map

In [ ]:
priority = visibility_summary.sort_values(['n_visible_nights', 'best_observable_minutes'], ascending=False)
priority_csv, priority_png = save_priority_products(priority, OUTPUT / 'tables', title='Observing priority summary', subtitle='Targets passing the configured magnitude and local-visibility criteria.')
sky_map = save_target_sky_map(selected, OUTPUT / 'sky_map.png', title='Selected targets')
display(priority.head())

## 6. Optional light curves and LSST image products

The light-curve call is optional because it may require provider credentials. LSST coverage and coadd cutouts are requested only for selected targets when enabled.

In [ ]:
lightcurve_pngs = []
if MAKE_LIGHTCURVES and not selected.empty:
    from target_selection import report
    lightcurve_pngs = report.create_lightcurve_plots(selected, output_dir=OUTPUT / 'lightcurves', observatory_photometry=observations, layers=('mop_photometry', 'opm_photometry'), observation_date=START_DATE, overwrite=True)
print('Light-curve PNGs:', len(lightcurve_pngs))

lsst_cutout_manifest = None
if MAKE_LSST_CUTOUTS and not selected.empty:
    from target_selection import data_source
    lsst_coverage = data_source.lsst.coverage(selected, image_kinds=('individual', 'coadd'), properties=('maglim', 'psf_size', 'exposure_time'))
    lsst_coverage.to_csv(OUTPUT / 'lsst_coverage.csv', index=False)
    lsst_cutout_manifest = data_source.lsst.get_image(selected.merge(lsst_coverage, on='target_id', how='left'), kind='coadd', output_dir=OUTPUT / 'cutouts' / 'png', fits_dir=OUTPUT / 'cutouts' / 'fits', bands=tuple('ugrizy'), add_rgb=False)

## 7. Assemble the PDF

In [ ]:
cutout_images = sorted((OUTPUT / 'cutouts' / 'png').glob('*.png')) if MAKE_LSST_CUTOUTS else []
pdf_path = save_planning_pdf(OUTPUT / 'seasonal_observing_plan.pdf', sky_map=sky_map, table_images=(priority_png, visibility_png), visibility_images=visibility_plots, lightcurve_images=lightcurve_pngs, cutout_images=cutout_images)
print(pdf_path)